# Graph feature evaluation

This notebook tests whether transaction-network features improve the 75% HistGradientBoosting and 25% CatBoost ensemble from notebook 04. Earlier degree-based features showed no consistent benefit, so this experiment considers PageRank, neighbour connectivity and measures of component cohesion.

The edge list does not establish a reliable transaction direction or confirm when each connection becomes available. We therefore treat it as an undirected network and calculate structural features separately for each time step. These features use no fraud labels, but they assume that connections within the prediction time step are observable. The experiment is exploratory rather than evidence that the features would be available in a live fraud detection system.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from catboost import CatBoostClassifier

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA, OUT = ROOT/'data'/'raw', ROOT/'submissions'
train = pd.read_csv(DATA/'train.csv')
test = pd.read_csv(DATA/'test.csv')
edges = pd.read_csv(DATA/'txs_edgelist.csv')
sample = pd.read_csv(DATA/'sample_submission.csv')
ID, TIME, LABEL = 'txId', 'time_step', 'label'
assert {ID,TIME,LABEL}.issubset(train.columns)
assert {ID,TIME}.issubset(test.columns)
assert {'txId1','txId2'}.issubset(edges.columns)
assert train[ID].is_unique and test[ID].is_unique
assert not set(train[ID]).intersection(test[ID])
features = [c for c in train if c.startswith('feat_')]
assert features and all(c in test for c in features)
labelled = train.dropna(subset=[LABEL]).copy()
print('train/test/edges:',train.shape,test.shape,edges.shape)
print('labelled:',len(labelled),'raw features:',len(features))

train/test/edges: (141772, 168) (15329, 168) (234355, 2)
labelled: 31235 raw features: 165


## Graph coverage

Before calculating network features, we check how many edges connect transactions present in the supplied datasets and whether their endpoints belong to the same time step. An endpoint missing from the transaction tables is not necessarily absent from the underlying network; the analysis covers only the observed portion of the graph.

In [2]:
all_nodes = pd.concat([train[[ID,TIME]],test[[ID,TIME]]],ignore_index=True)
assert all_nodes[ID].is_unique
node_time = all_nodes.set_index(ID)[TIME]
e = edges[['txId1','txId2']].copy()
e['t1'] = e.txId1.map(node_time)
e['t2'] = e.txId2.map(node_time)
e['both_observed'] = e.t1.notna() & e.t2.notna()
e['same_step'] = e.both_observed & e.t1.eq(e.t2)
print(pd.Series({'total_edges':len(e),'both_endpoints_observed':int(e.both_observed.sum()),
                 'same_step_observed':int(e.same_step.sum()),
                 'different_step_observed':int((e.both_observed & ~e.same_step).sum()),
                 'unknown_endpoint':int((~e.both_observed).sum()),
                 'reverse_pairs':len(set(zip(e.txId1,e.txId2)) & set(zip(e.txId2,e.txId1)))}))
usable = e.loc[e.same_step,['txId1','txId2','t1']].copy()
usable['t1'] = usable.t1.astype(int)
print('Known train/test time ranges:',(train[TIME].min(),train[TIME].max()),(test[TIME].min(),test[TIME].max()))

total_edges                234355
both_endpoints_observed    175918
same_step_observed         175918
different_step_observed         0
unknown_endpoint            58437
reverse_pairs                   0
dtype: int64
Known train/test time ranges: (np.int64(1), np.int64(35)) (np.int64(36), np.int64(49))


## Network features

For each time step, the observed transactions and their connections form an undirected graph. We calculate PageRank, the mean and maximum degree of each node's neighbours, component density, k-core number and triangle count. Degree and component size are also retained for comparison.

PageRank measures structural prominence within the observed time-step graph; it is not seeded with fraud labels. Component density describes how closely connected the observed component is, while k-core number and triangle count capture local cohesion. We avoid using component identifiers as features because their arbitrary values would not be comparable across time steps.

The resulting feature distributions differ substantially between train and test transactions, particularly component size and density. This is a reason to validate graph features cautiously rather than assume that additional network information will improve prediction.

In [3]:
records=[]
for t, frame in all_nodes.groupby(TIME,sort=True):
    ids=frame[ID].tolist()
    G=nx.Graph()
    G.add_nodes_from(ids)
    sub=usable.loc[usable.t1.eq(t),['txId1','txId2']]
    G.add_edges_from(sub.itertuples(index=False,name=None))
    G.remove_edges_from(nx.selfloop_edges(G))
    deg=dict(G.degree())
    # PageRank on the undirected observed graph; normalised by node count for comparability across steps.
    pr=nx.pagerank(G,alpha=0.85,max_iter=200,tol=1e-7) if G.number_of_nodes()>0 else {}
    core=nx.core_number(G)
    tri=nx.triangles(G)
    component_size={};component_density={}
    for comp in nx.connected_components(G):
        nodes=list(comp); size=len(nodes)
        density=nx.density(G.subgraph(nodes)) if size>1 else 0.0
        for node in nodes:
            component_size[node]=size
            component_density[node]=density
    for node in ids:
        neigh=list(G.neighbors(node))
        degrees=[deg[x] for x in neigh]
        records.append((node,deg[node],component_size[node],pr[node]*len(ids),
                        float(np.mean(degrees)) if degrees else 0.0,
                        max(degrees,default=0),component_density[node],
                        core[node],tri[node]))
    print(f'time {t}: {len(ids):,} nodes, {G.number_of_edges():,} edges',end='\r')
GRAPH_COLS=['graph_degree','graph_component_size','graph_pagerank_scaled',
            'graph_neigh_mean_degree','graph_neigh_max_degree',
            'graph_component_density','graph_core_number','graph_triangle_count']
graph_df=pd.DataFrame(records,columns=[ID]+GRAPH_COLS)
assert graph_df[ID].is_unique and len(graph_df)==len(all_nodes)
train_g=train.merge(graph_df,on=ID,how='left',validate='one_to_one')
test_g=test.merge(graph_df,on=ID,how='left',validate='one_to_one')
assert train_g[GRAPH_COLS].notna().all().all() and test_g[GRAPH_COLS].notna().all().all()
display(train_g[GRAPH_COLS].describe().T)
display(pd.DataFrame({'train_mean':train_g[GRAPH_COLS].mean(),
                      'test_mean':test_g[GRAPH_COLS].mean()}))

,count,mean,std,min,25%,50%,75%,max
graph_degree,141772.0,2.302204,4.045061,1.000000,1.000000,2.000000,2.000000,473.000000
graph_component_size,141772.0,4702.339418,1603.087709,1089.000000,3537.000000,4525.000000,5894.000000,7880.000000
graph_pagerank_scaled,141772.0,1.000000,1.365382,0.229448,0.578176,0.932762,1.057409,148.321618
graph_neigh_mean_degree,141772.0,11.265520,28.745028,1.058824,2.000000,2.666667,6.500000,473.000000
graph_neigh_max_degree,141772.0,17.213272,44.892692,2.000000,2.000000,3.000000,10.000000,473.000000
graph_component_density,141772.0,0.000556,0.000232,0.000295,0.000404,0.000505,0.000625,0.001972
graph_core_number,141772.0,1.400016,0.680616,1.000000,1.000000,1.000000,2.000000,9.000000
graph_triangle_count,141772.0,0.154283,1.831062,0.000000,0.000000,0.000000,0.000000,74.000000


,train_mean,test_mean
graph_degree,2.302204,1.660121
graph_component_size,4702.339418,432.458412
graph_pagerank_scaled,1.000000,1.000000
graph_neigh_mean_degree,11.265520,7.394320
graph_neigh_max_degree,17.213272,9.486855
graph_component_density,0.000556,0.073933
graph_core_number,1.400016,0.953552
graph_triangle_count,0.154283,0.084741


## Model and feature configurations

All comparisons use the 04 notebook ensemble without changing its model settings or blend weights: 75% HistGradientBoosting and 25% CatBoost averaged across five random seeds. The reference uses the 165 original transaction features. Three alternatives add PageRank and neighbour statistics, cohesion statistics, or all six new network features.

In [4]:
PERIODS=[(15,19),(20,24),(25,29)]
SEEDS=[11,42,73,101,202]
CAT_PARAMS=dict(iterations=700,depth=6,learning_rate=0.045,l2_leaf_reg=5,
                loss_function='Logloss',verbose=False,thread_count=-1)
FEATURE_SETS={
    'reference_raw':features,
    'pagerank_neighbours':features+['graph_pagerank_scaled','graph_neigh_mean_degree','graph_neigh_max_degree'],
    'cohesion_core':features+['graph_component_density','graph_core_number','graph_triangle_count'],
    'all_new_graph':features+['graph_pagerank_scaled','graph_neigh_mean_degree','graph_neigh_max_degree',
                              'graph_component_density','graph_core_number','graph_triangle_count'],
}
def fit_hgb(x,y):
    m=HistGradientBoostingClassifier(max_iter=300,learning_rate=0.05,max_leaf_nodes=31,
                                     l2_regularization=1.0,random_state=42)
    return m.fit(x,y)
def fit_cat(x,y,seed):
    return CatBoostClassifier(**CAT_PARAMS,random_seed=seed).fit(x,y)
def metrics(y,p):
    return {'pr_auc':average_precision_score(y,p),'roc_auc':roc_auc_score(y,p)}
def predict_frozen_blend(tr,va,cols):
    y=tr[LABEL].astype(int)
    ph=fit_hgb(tr[cols],y).predict_proba(va[cols])[:,1]
    pc=np.column_stack([fit_cat(tr[cols],y,s).predict_proba(va[cols])[:,1] for s in SEEDS]).mean(axis=1)
    return .75*ph+.25*pc
print('Feature counts:',{k:len(v) for k,v in FEATURE_SETS.items()})

Feature counts: {'reference_raw': 165, 'pagerank_neighbours': 168, 'cohesion_core': 168, 'all_new_graph': 171}


## Chronological validation

We evaluate each feature set on steps 15–19, 20–24 and 25–29, always training on earlier labelled transactions. Four feature sets, three validation windows and six base-model fits per comparison require 72 model fits. The evaluation uses average precision (PR-AUC) as the main comparison metric and also records ROC-AUC.

In [5]:
results=[]; val_predictions={}
for lo,hi in PERIODS:
    tr=train_g.loc[(train_g[TIME]<lo)&train_g[LABEL].notna()]
    va=train_g.loc[train_g[TIME].between(lo,hi)&train_g[LABEL].notna()]
    assert tr[LABEL].nunique()==2 and va[LABEL].nunique()==2
    period=f'{lo}-{hi}'
    print(f'\nPeriod {period}: train {len(tr):,}, validation {len(va):,}, fraud prevalence {va[LABEL].mean():.3f}',flush=True)
    val_predictions[period]={}
    for name,cols in FEATURE_SETS.items():
        p=predict_frozen_blend(tr,va,cols)
        val_predictions[period][name]=p
        row={'period':period,'model':name,'n_train':len(tr),'n_valid':len(va),
             'fraud_rate':va[LABEL].mean(),**metrics(va[LABEL].astype(int),p)}
        results.append(row)
        print(name,round(row['pr_auc'],6),round(row['roc_auc'],6),flush=True)
results=pd.DataFrame(results)
display(results.pivot(index='model',columns='period',values='pr_auc'))


Period 15-19: train 14,896, validation 3,093, fraud prevalence 0.164
reference_raw 0.98827 0.995174
pagerank_neighbours 0.988132 0.995106
cohesion_core 0.988252 0.995165
all_new_graph 0.988575 0.99545

Period 20-24: train 17,989, validation 5,617, fraud prevalence 0.126
reference_raw 0.878462 0.946498
pagerank_neighbours 0.871845 0.937089
cohesion_core 0.876809 0.944731
all_new_graph 0.873724 0.941347

Period 25-29: train 23,606, validation 2,775, fraud prevalence 0.235
reference_raw 0.993123 0.997625
pagerank_neighbours 0.991607 0.997236
cohesion_core 0.99206 0.997313
all_new_graph 0.992458 0.997441


period,15-19,20-24,25-29
model,,,
all_new_graph,0.988575,0.873724,0.992458
cohesion_core,0.988252,0.876809,0.992060
pagerank_neighbours,0.988132,0.871845,0.991607
reference_raw,0.988270,0.878462,0.993123


## Comparison with the original features

The raw-feature ensemble achieved PR-AUC values of 0.988270, 0.878462 and 0.993123 across the three validation windows. None of the graph-feature groups improved the mean PR-AUC. The cohesion features had the smallest mean decline (−0.000911), while the PageRank and neighbour group had the largest (−0.002757).

All three alternatives performed worse on steps 20–24, the weakest period for the reference model. The combined graph features improved steps 15–19 slightly (+0.000305) but reduced performance in the other two windows. We therefore retain the original feature set.

These validation periods have been used in earlier experiments, so the comparisons are useful for screening but are not an independent estimate of future performance.

In [6]:
base=results.loc[results.model.eq('reference_raw')].set_index('period')
comparison=[]
for name in FEATURE_SETS:
    if name=='reference_raw':continue
    cand=results.loc[results.model.eq(name)].set_index('period').loc[base.index]
    delta=cand.pr_auc-base.pr_auc
    comparison.append({'model':name,'mean_pr_auc':cand.pr_auc.mean(),
                       'mean_delta_pr':delta.mean(),'delta_15_19':delta.loc['15-19'],
                       'delta_20_24':delta.loc['20-24'],'delta_25_29':delta.loc['25-29'],
                       'positive_periods':int((delta>0).sum()),
                       'eligible':bool(delta.mean()>0 and delta.loc['20-24']>=0 and (delta>0).sum()>=2)})
comparison=pd.DataFrame(comparison).sort_values('mean_delta_pr',ascending=False)
display(comparison)
eligible=comparison.loc[comparison.eligible]
chosen=None if eligible.empty else eligible.iloc[0]['model']
print('Selected challenger:',chosen if chosen else 'NONE — retain notebook_04')
print('Caution: selection uses reused validation windows; do not assume Kaggle improvement.')

,model,mean_pr_auc,mean_delta_pr,delta_15_19,delta_20_24,delta_25_29,positive_periods,eligible
1,cohesion_core,0.952374,-0.000911,-0.000018,-0.001653,-0.001063,0,False
2,all_new_graph,0.951586,-0.001700,0.000305,-0.004738,-0.000665,1,False
0,pagerank_neighbours,0.950528,-0.002757,-0.000139,-0.006617,-0.001516,0,False


Selected challenger: NONE — retain notebook_04
Caution: selection uses reused validation windows; do not assume Kaggle improvement.


## Submission generation

Submission generation is disabled because none of the graph-feature groups passed the validation comparison. The code is retained for reproducibility but should only be enabled after confirming that the required graph edges would be available for test transactions. notebook 04's public leaderboard score was 0.94756.

In [7]:
MAKE_SUBMISSION=False
if not MAKE_SUBMISSION:
    print('Submission disabled. Inspect validation results and graph coverage first.')
elif chosen is None:
    print('No eligible challenger. No submission generated.')
else:
    OUT.mkdir(parents=True,exist_ok=True)
    dest=OUT/'experiment16_advanced_graph.csv'
    if dest.exists():
        raise FileExistsError(f'Existing submission will not be overwritten: {dest}')
    cols=FEATURE_SETS[chosen]
    tr=train_g.loc[train_g[LABEL].notna()]
    p=predict_frozen_blend(tr,test_g,cols)
    assert sample.columns.tolist()==['index','target']
    assert len(sample)==len(p) and np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
    submission=sample.copy();submission['target']=p
    submission.to_csv(dest,index=False)
    print('Saved',dest,'using',chosen)
    display(submission.head())

Submission disabled. Inspect validation results and graph coverage first.


## Findings and limitations

The additional network features did not produce a reliable improvement over the original transaction features. Their distributions also changed markedly between the training and test periods, which may limit their usefulness under temporal shift.

The analysis is restricted to edges whose endpoints appear in the supplied transaction tables. It treats the graph as undirected because edge direction has not been verified, and it assumes access to connections within each time step. We did not use neighbour fraud rates or fraud-seeded PageRank: those would require careful, fold-specific handling of historical labels to avoid leakage. No Kaggle submission was generated from this experiment.